# DQT Daily Dial Miss Decomposition

Quantify **daily dial miss** (SARIMA `y_hat_sarima_cal` vs shipped `alt_50`) on the full long-lead paint cohort.

```bash
uv run python scripts/run_dial_miss_decomposition.py
```

Artifacts: `data/etp/dial-miss/` · Exec memo: `documentation/etp-slider/dqt-dial-miss-exec-readout.md`

In [1]:
from pathlib import Path

import polars as pl
from IPython.display import display

from dqt import repo_root, resolve_data_dir
from dqt.dial_miss.decomposition import run_dial_miss_decomposition

DATA = resolve_data_dir()
OUT = DATA / "etp" / "dial-miss"

In [2]:
result = run_dial_miss_decomposition(data_dir=DATA, repo_root=repo_root())
print(f"Loads with dial miss: {result['n_loads']:,}")
print("Hypotheses:", result["manifest"]["hypotheses"])

Loads with dial miss: 230,405
Hypotheses: {'H1': 'Confirmed — dial miss magnitude non-trivial at book', 'H2': 'Confirmed — dial miss explains <15% of avail→48hr shift', 'H3': 'Inconclusive', 'H4': 'Confirmed — full leveling improves att50 with bounded MAE', 'H5': 'Rejected — S1 does not match full leveling benefit'}


In [3]:
daily = pl.read_parquet(OUT / "daily-dial-miss.parquet")
load_miss = pl.read_parquet(OUT / "load-dial-miss.parquet")
display(daily.select("booked_date", "dial_miss_pp", "alt_50", "y_hat_sarima_cal").tail(10))
display(load_miss.select("loadnumber", "dial_miss_pp", "dial_miss_usd").head(5))

booked_date,dial_miss_pp,alt_50,y_hat_sarima_cal
date,f64,f64,f64
2026-08-17,0.532046,0.421517,0.426838
2026-08-18,2.886682,0.42221,0.451077
2026-08-19,4.061122,0.423217,0.463828
2026-08-20,2.308767,0.424499,0.447586
2026-08-21,5.108771,0.426531,0.477619
2026-08-24,1.275022,0.432306,0.445056
2026-08-25,4.626259,0.43441,0.480673
2026-08-26,7.093674,0.437046,0.507983
2026-08-27,7.690305,0.441145,0.518048


loadnumber,dial_miss_pp,dial_miss_usd
i32,f64,f64
9023538,1.245555,2.103588
8678424,-1.28946,-4.688672
8193678,-1.545753,-1.999363
7952179,3.354869,19.365135
6747511,4.743328,10.662703


In [4]:
cohort = pl.read_csv(OUT / "cohort-dial-miss.csv")
frontier = pl.read_csv(OUT / "leveling-frontier.csv")
display(cohort.sort("mean_dial_miss_usd", descending=True).head(12))
display(frontier)

dimension,segment,n,mean_dial_miss_pp,mean_dial_miss_usd,att50_at_48hr,mae_at_48hr_s1,est_dial_attrib_mae
str,str,i64,f64,f64,f64,f64,f64
"""dial_miss_quintile""","""Q5""",46081,7.0586,37.47,0.4611,222.94,27.25
"""dial_miss_sign""","""over""",144554,4.9514,20.77,0.4706,NaN,15.1
"""book_dow""","""5""",38645,4.2524,18.61,0.4646,178.06,13.53
"""booking_window_hrs""","""226""",339,3.0744,17.06,0.4533,184.15,12.41
"""booking_window_hrs""","""228""",185,2.769,15.42,0.4083,195.93,11.21
…,…,…,…,…,…,…,…
"""booking_window_hrs""","""254""",113,2.1027,14.47,0.4286,158.46,10.52
"""booking_window_hrs""","""225""",651,2.6962,13.97,0.4472,179.13,10.16
"""booking_window_hrs""","""230""",128,2.8266,13.84,0.4545,179.7,10.07


policy,alpha,att50,mae
str,f64,f64,f64
"""status_quo""",0.0,0.486404,134.670262
"""partial_25""",0.25,0.491838,134.281266
"""partial_50""",0.5,0.497528,134.060462
"""partial_75""",0.75,0.502893,134.027336
"""full_leveling""",1.0,0.508309,134.202992
